In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report

np.random.seed(42)
torch.manual_seed(42)

In [ ]:
from google.colab import files
uploaded = files.upload()
df = pd.read_csv(next(iter(uploaded)))
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.drop(columns='sample_id').corr()['safe'].sort_values(ascending=False)

In [ ]:
FEATURES = ['ph', 'tds', 'temp']
X = df[FEATURES]
y = df['safe']
groups = df['sample_id']

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=3)
train_idx, test_idx = next(splitter.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

print('X_train shape --', X_train.shape)
print('y_train shape --', y_train.shape)
print('X_test shape --', X_test.shape)
print('y_test shape --', y_test.shape)

In [ ]:
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
X_train_num = num_pipeline.fit_transform(X_train)
X_test_num = num_pipeline.transform(X_test)

In [ ]:
def show_results(y_true, y_pred, title):
    print(classification_report(y_true, y_pred, target_names=['unsafe', 'safe']))
    plt.figure(figsize=(6, 4))
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Pastel1',
                xticklabels=['Predicted unsafe', 'Predicted safe'],
                yticklabels=['Actual unsafe', 'Actual safe'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(title)
    plt.show()

SVM

In [ ]:
grid = GridSearchCV(
    SVC(class_weight='balanced'),
    {'kernel': ['rbf', 'poly'], 'C': [0.1, 1, 10, 100], 'gamma': ['scale', 0.1, 1]},
    cv=GroupKFold(n_splits=5), scoring='balanced_accuracy', n_jobs=-1)
grid.fit(X_train_num, y_train, groups=groups_train)

print(grid.best_params_)
show_results(y_test, grid.predict(X_test_num), 'SVM')

In [ ]:
joblib.dump((num_pipeline, grid.best_estimator_), 'svm_water.joblib')

Deep models (each of the 3 features is fed as one step of a sequence)

In [ ]:
X_train_t = torch.tensor(X_train_num, dtype=torch.float32).unsqueeze(-1)
X_test_t = torch.tensor(X_test_num, dtype=torch.float32).unsqueeze(-1)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)

pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()], dtype=torch.float32)


class GRUModel(nn.Module):
    def __init__(self, hidden_size=64):
        super().__init__()
        self.gru = nn.GRU(1, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])


class LSTMModel(nn.Module):
    def __init__(self, hidden_size=64):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


class HybridModel(nn.Module):
    def __init__(self, hidden_size=64):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden_size, batch_first=True)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out, _ = self.gru(out)
        return self.fc(out[:, -1, :])


def train_and_test(model, title, epochs=300, lr=0.01):
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        loss = criterion(model(X_train_t), y_train_t)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if (epoch + 1) % 50 == 0:
            print(f'Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}')
    with torch.no_grad():
        y_pred = (torch.sigmoid(model(X_test_t)) >= 0.5).int().squeeze().numpy()
    show_results(y_test, y_pred, title)

GRU

In [ ]:
train_and_test(GRUModel(), 'GRU')

LSTM

In [ ]:
train_and_test(LSTMModel(), 'LSTM')

Hybrid GRU+LSTM

In [ ]:
train_and_test(HybridModel(), 'Hybrid GRU+LSTM')